## Load the data

In [ ]:
import pandas
ratings = pandas.read_csv("ratings.csv")[["userId", "movieId", "rating"]]
ratings.head()

,userId,movieId,rating
0,1,31,2.5
1,1,1029,3.0
2,1,1061,3.0
3,1,1129,2.0
4,1,1172,4.0


## Create the dataset

In [7]:
import numpy as np
from scipy.sparse import csr_matrix

# Build a sparse user-item matrix from the ratings data
# Rows = users (0-indexed), Columns = movies (0-indexed), Values = ratings
user_item_matrix = csr_matrix(
    (ratings['rating'].values,           # non-zero values
     (ratings['userId'].values - 1,      # row indices (userId is 1-based, so subtract 1)
      ratings['movieId'].values - 1)),   # col indices (movieId is 1-based, so subtract 1)
)

print(f"Number of users: {user_item_matrix.shape[0]}")
print(f"Number of movies: {user_item_matrix.shape[1]}")
print(f"Number of ratings: {user_item_matrix.nnz}")
user_item_matrix

Number of users: 671
Number of movies: 163949
Number of ratings: 100004


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 100004 stored elements and shape (671, 163949)>

## Build the trainset

In [9]:
# user_item_matrix already serves as the full trainset
trainset = user_item_matrix

In [20]:
# Convert the sparse matrix to COO (Coordinate) format to iterate over non-zero entries.
# COO format stores each rating as three arrays: row (user), col (movie), data (rating value).
coo = user_item_matrix.tocoo()

# Zip the three arrays together to get (user_idx, movie_idx, rating) tuples.
# At this point, user_idx = userId - 1 and movie_idx = movieId - 1 (0-based indices).
all_ratings = list(zip(coo.row, coo.col, coo.data))

# Sort the list by (user_idx, movie_idx) in ascending order.
# Also cast numpy types (np.int32, np.float64) to plain Python int/float
# to avoid verbose output like np.int32(0) when printing.
# Tuples are sorted lexicographically, so (user, movie) order is guaranteed.
sorted((int(u), int(i), float(r)) for u, i, r in all_ratings)

[(0, 30, 2.5),
 (0, 1028, 3.0),
 (0, 1060, 3.0),
 (0, 1128, 2.0),
 (0, 1171, 4.0),
 (0, 1262, 2.0),
 (0, 1286, 2.0),
 (0, 1292, 2.0),
 (0, 1338, 3.5),
 (0, 1342, 2.0),
 (0, 1370, 2.5),
 (0, 1404, 1.0),
 (0, 1952, 4.0),
 (0, 2104, 4.0),
 (0, 2149, 3.0),
 (0, 2192, 2.0),
 (0, 2293, 2.0),
 (0, 2454, 2.5),
 (0, 2967, 1.0),
 (0, 3670, 3.0),
 (1, 9, 4.0),
 (1, 16, 5.0),
 (1, 38, 5.0),
 (1, 46, 4.0),
 (1, 49, 4.0),
 (1, 51, 3.0),
 (1, 61, 3.0),
 (1, 109, 4.0),
 (1, 143, 3.0),
 (1, 149, 5.0),
 (1, 152, 4.0),
 (1, 160, 3.0),
 (1, 164, 3.0),
 (1, 167, 3.0),
 (1, 184, 3.0),
 (1, 185, 3.0),
 (1, 207, 3.0),
 (1, 221, 5.0),
 (1, 222, 1.0),
 (1, 224, 3.0),
 (1, 234, 3.0),
 (1, 247, 3.0),
 (1, 252, 4.0),
 (1, 260, 4.0),
 (1, 264, 5.0),
 (1, 265, 5.0),
 (1, 271, 3.0),
 (1, 272, 4.0),
 (1, 291, 3.0),
 (1, 295, 4.0),
 (1, 299, 3.0),
 (1, 313, 4.0),
 (1, 316, 2.0),
 (1, 318, 1.0),
 (1, 338, 3.0),
 (1, 348, 4.0),
 (1, 349, 4.0),
 (1, 355, 3.0),
 (1, 356, 3.0),
 (1, 363, 3.0),
 (1, 366, 3.0),
 (1, 369, 2.0)

In [21]:
user_to_inner = {}
movie_to_inner = {}

result = []
# iterrows() returns (index, row) pairs.
# _ is used instead of a variable name to signal that the index is intentionally unused.
for _, row in ratings.iterrows():
    uid = int(row['userId'])
    mid = int(row['movieId'])
    r = float(row['rating'])
    
    if uid not in user_to_inner:
        user_to_inner[uid] = len(user_to_inner)
    if mid not in movie_to_inner:
        movie_to_inner[mid] = len(movie_to_inner)
    
    result.append((user_to_inner[uid], movie_to_inner[mid], r))

result

[(0, 0, 2.5),
 (0, 1, 3.0),
 (0, 2, 3.0),
 (0, 3, 2.0),
 (0, 4, 4.0),
 (0, 5, 2.0),
 (0, 6, 2.0),
 (0, 7, 2.0),
 (0, 8, 3.5),
 (0, 9, 2.0),
 (0, 10, 2.5),
 (0, 11, 1.0),
 (0, 12, 4.0),
 (0, 13, 4.0),
 (0, 14, 3.0),
 (0, 15, 2.0),
 (0, 16, 2.0),
 (0, 17, 2.5),
 (0, 18, 1.0),
 (0, 19, 3.0),
 (1, 20, 4.0),
 (1, 21, 5.0),
 (1, 22, 5.0),
 (1, 23, 4.0),
 (1, 24, 4.0),
 (1, 25, 3.0),
 (1, 26, 3.0),
 (1, 27, 4.0),
 (1, 28, 3.0),
 (1, 29, 5.0),
 (1, 30, 4.0),
 (1, 31, 3.0),
 (1, 32, 3.0),
 (1, 33, 3.0),
 (1, 34, 3.0),
 (1, 35, 3.0),
 (1, 36, 3.0),
 (1, 37, 5.0),
 (1, 38, 1.0),
 (1, 39, 3.0),
 (1, 40, 3.0),
 (1, 41, 3.0),
 (1, 42, 4.0),
 (1, 43, 4.0),
 (1, 44, 5.0),
 (1, 45, 5.0),
 (1, 46, 3.0),
 (1, 47, 4.0),
 (1, 48, 3.0),
 (1, 49, 4.0),
 (1, 50, 3.0),
 (1, 51, 4.0),
 (1, 52, 2.0),
 (1, 53, 1.0),
 (1, 54, 3.0),
 (1, 55, 4.0),
 (1, 56, 4.0),
 (1, 57, 3.0),
 (1, 58, 3.0),
 (1, 59, 3.0),
 (1, 60, 3.0),
 (1, 61, 2.0),
 (1, 62, 3.0),
 (1, 63, 3.0),
 (1, 64, 3.0),
 (1, 65, 3.0),
 (1, 66, 2.0),
 (1, 